# 01 - TensorFlow/Keras Regularization and Generalization A/B Tests

This notebook compares several regularization and generalization strategies on a compact image classification task.

## Topics covered
- L1 / L2 regularization
- dropout
- early stopping
- Monte Carlo dropout
- initialization experiments
- batch normalization
- custom dropout
- custom regularization
- callbacks and TensorBoard

The experiments are intentionally lightweight so they fit a Colab workflow and are easy to explain during a video walkthrough.


In [ ]:
# Colab setup
%pip -q install -U tensorflow tensorboard pandas matplotlib scikit-learn


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("TensorFlow version:", tf.__version__)


## Dataset

We use Fashion-MNIST because it is small, fast to train, and still noisy enough to make regularization effects visible.


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# keep the notebook fast for Colab demos
x_train = x_train[:15000]
y_train = y_train[:15000]
x_test = x_test[:3000]
y_test = y_test[:3000]

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

val_size = 3000
x_val, y_val = x_train[-val_size:], y_train[-val_size:]
x_tr, y_tr = x_train[:-val_size], y_train[:-val_size]

print("train:", x_tr.shape, y_tr.shape)
print("val:", x_val.shape, y_val.shape)
print("test:", x_test.shape, y_test.shape)


In [ ]:
class MCDropout(layers.Dropout):
    # Dropout that can remain active at inference time for uncertainty estimation.
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

class StochasticScaleDropout(layers.Layer):
    '''
    A custom dropout-like layer that randomly rescales activations.
    This is not meant to replace standard dropout in production;
    it exists here to demonstrate how to create a custom regularization layer.
    '''
    def __init__(self, rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.uniform(tf.shape(inputs), 1.0 - self.rate, 1.0 + self.rate)
            return inputs * noise
        return inputs

    def get_config(self):
        return {"rate": self.rate, **super().get_config()}

class ActivationEnergyRegularizer(keras.regularizers.Regularizer):
    # Penalizes mean squared activation magnitude.
    def __init__(self, strength=1e-4):
        self.strength = strength

    def __call__(self, x):
        return self.strength * tf.reduce_mean(tf.square(x))

    def get_config(self):
        return {"strength": float(self.strength)}


## Model factory

We keep the backbone simple and expose regularization options as arguments so A/B tests are easy to run.


In [ ]:
def get_initializer(name):
    mapping = {
        "glorot_uniform": keras.initializers.GlorotUniform(seed=SEED),
        "he_normal": keras.initializers.HeNormal(seed=SEED),
        "lecun_normal": keras.initializers.LecunNormal(seed=SEED),
    }
    return mapping[name]

def build_model(
    hidden_units=(256, 128),
    dropout_rate=0.0,
    use_batchnorm=False,
    kernel_reg=None,
    activity_reg=None,
    initializer="glorot_uniform",
    custom_dropout=False,
    mc_dropout=False,
):
    model = keras.Sequential(name="regularization_abtest")
    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Flatten())

    for units in hidden_units:
        model.add(
            layers.Dense(
                units,
                kernel_initializer=get_initializer(initializer),
                kernel_regularizer=kernel_reg,
                activity_regularizer=activity_reg,
                use_bias=not use_batchnorm,
            )
        )
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.ReLU())

        if dropout_rate > 0:
            if mc_dropout:
                model.add(MCDropout(dropout_rate))
            elif custom_dropout:
                model.add(StochasticScaleDropout(dropout_rate))
            else:
                model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation="softmax"))
    return model


In [ ]:
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def fit_experiment(name, build_kwargs, epochs=12):
    log_dir = os.path.join("logs", "part1_tf", name)
    os.makedirs(log_dir, exist_ok=True)

    model = compile_model(build_model(**build_kwargs))

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, verbose=1
        ),
        keras.callbacks.TensorBoard(log_dir=log_dir),
    ]

    history = model.fit(
        x_tr,
        y_tr,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=128,
        verbose=0,
        callbacks=callbacks,
    )

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    result = {
        "experiment": name,
        "best_val_acc": float(np.max(history.history["val_accuracy"])),
        "test_acc": float(test_acc),
        "epochs_ran": len(history.history["loss"]),
    }
    return model, history, result


## Core A/B tests


In [ ]:
experiments = {
    "baseline": dict(),
    "l2": dict(kernel_reg=regularizers.l2(1e-4)),
    "l1_l2": dict(kernel_reg=regularizers.l1_l2(l1=1e-5, l2=1e-4)),
    "dropout": dict(dropout_rate=0.30),
    "batchnorm": dict(use_batchnorm=True),
    "dropout_batchnorm": dict(dropout_rate=0.30, use_batchnorm=True),
    "custom_regularizer": dict(activity_reg=ActivationEnergyRegularizer(1e-4)),
    "custom_dropout": dict(dropout_rate=0.20, custom_dropout=True),
}

trained_models = {}
histories = {}
rows = []

for name, kwargs in experiments.items():
    print(f"Running: {name}")
    model, history, row = fit_experiment(name, kwargs, epochs=10)
    trained_models[name] = model
    histories[name] = history
    rows.append(row)

results_df = pd.DataFrame(rows).sort_values("test_acc", ascending=False)
results_df


In [ ]:
plt.figure(figsize=(10, 5))
for name, history in histories.items():
    plt.plot(history.history["val_accuracy"], label=name)
plt.title("Validation accuracy by experiment")
plt.xlabel("Epoch")
plt.ylabel("val_accuracy")
plt.legend()
plt.show()


## Initialization comparison

Very roughly:

- **He initialization** is a strong default for ReLU-family activations.
- **Glorot/Xavier** is a common general-purpose default.
- **LeCun** is often paired with SELU-style self-normalizing networks.

We keep the activation fixed to ReLU here so the A/B result is easy to interpret.


In [ ]:
init_rows = []
for init_name in ["glorot_uniform", "he_normal", "lecun_normal"]:
    model, history, row = fit_experiment(
        f"init_{init_name}",
        dict(initializer=init_name, dropout_rate=0.2),
        epochs=8,
    )
    init_rows.append(row)

pd.DataFrame(init_rows).sort_values("test_acc", ascending=False)


## Monte Carlo dropout for uncertainty

The trick: leave dropout active at inference time, run the same input many times, then measure prediction variance.


In [ ]:
mc_model, mc_history, mc_row = fit_experiment(
    "mc_dropout_model",
    dict(dropout_rate=0.30, mc_dropout=True),
    epochs=10,
)
mc_row


In [ ]:
sample_batch = x_test[:16]
sample_labels = y_test[:16]

mc_probs = []
for _ in range(25):
    mc_probs.append(mc_model(sample_batch, training=True).numpy())
mc_probs = np.stack(mc_probs)

mean_probs = mc_probs.mean(axis=0)
std_probs = mc_probs.std(axis=0)
preds = mean_probs.argmax(axis=1)
uncertainty = std_probs.max(axis=1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, image, true_y, pred_y, u in zip(axes.ravel(), sample_batch, sample_labels, preds, uncertainty):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"T:{true_y} P:{pred_y}\nunc={u:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## TensorBoard

After training, launch TensorBoard inside Colab to inspect the logs.


In [ ]:
# In Colab, uncomment the next two lines after some experiments have run.
# %load_ext tensorboard
# %tensorboard --logdir logs/part1_tf
print("TensorBoard log root:", os.path.abspath("logs/part1_tf"))


## Takeaways

- L1/L2 help control weight growth.
- Dropout reduces co-adaptation between neurons.
- Early stopping acts like regularization because it prevents over-training.
- Batch normalization often stabilizes optimization and can improve generalization.
- MC dropout gives a cheap uncertainty estimate without training a separate ensemble.
- Initialization matters because it changes signal and gradient flow before learning even begins.

For the video, this notebook is ideal for showing baseline vs regularized models side by side.
